In [1]:
import gradio as gr
import os
import shutil
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core.memory import ChatMemoryBuffer

/Users/anveshradharapu/Library/Caches/pypoetry/virtualenvs/mlengineerprep-fSSVlCLJ-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
UPLOAD_DIR = "../data"
os.makedirs(UPLOAD_DIR, exist_ok=True)

In [3]:
embedding_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
llm = Ollama(model="gemma")
memory = ChatMemoryBuffer.from_defaults(token_limit=1500)

In [4]:
def build_index_from_pdf(file_path):
    reader = SimpleDirectoryReader(input_dir=UPLOAD_DIR)
    documents = reader.load_data()
    Settings.embed_model = embedding_model
    Settings.llm = llm
    index = VectorStoreIndex.from_documents(documents)
    query_engine = index.as_query_engine(chat_mode="context", memory=memory)
    return query_engine

In [5]:
def generate_icebreakers(file, tone, user_note):
    if not file:
        return "Please upload a LinkedIn profile PDF."
    
    # Save file
    shutil.copy(file.name, os.path.join(UPLOAD_DIR, "profile.pdf"))

    # Build index
    query_engine = build_index_from_pdf(os.path.join(UPLOAD_DIR, "profile.pdf"))

    # Generate prompt
    tone_prompts = {
        "Professional": "Generate 3 unique, insightful conversation starters that are formal and career-focused.",
        "Humorous": "Generate 3 witty and light-hearted icebreakers, but still relevant to this person’s career and interests.",
        "Inquisitive": "Generate 3 thoughtful and curious questions to spark an engaging conversation."
    }

    prompt = f"""{tone_prompts[tone]} Consider career highlights, interests, education, and fun facts. Extra note from user: {user_note}"""

    response = query_engine.query(prompt)
    return response.response

In [6]:
with gr.Blocks() as demo:
    gr.Markdown("## 🤝 AI Icebreaker Bot (Powered by LlamaIndex + Ollama Gemma)")
    
    with gr.Row():
        file_input = gr.File(label="Upload LinkedIn Profile (PDF)", file_types=[".pdf"])
        tone = gr.Radio(["Professional", "Humorous", "Inquisitive"], label="Select Tone", value="Professional")

    user_note = gr.Textbox(label="Optional Note (e.g. industry, event context)", placeholder="e.g. Tech networking event")

    output = gr.Textbox(label="Generated Icebreakers", lines=6)

    generate_btn = gr.Button("Generate Icebreakers")
    generate_btn.click(fn=generate_icebreakers, inputs=[file_input, tone, user_note], outputs=output)

demo.launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
